
<div class="alert alert-block alert-info">  
    
## Import & Connections
</div>

In [1]:
import numpy as np
import pandas as pd
import hashlib
import torch
import glob
import time
import os
import re
import tensorflow as tf
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from transformers import pipeline
from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

2025-01-23 02:43:18.208293: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_columns', 50)

In [3]:
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [4]:
start_date = '20250101'
end_date = '20250121'
city = 'Bangalore'

In [5]:
print(os.getcwd())
local_dataset = '/Users/E2074/local-datasets/customer/ads-monetisation/metadata'
print(local_dataset)

/Users/E2074/analytics/customer/Ads-Monetisation/metadata-analysis
/Users/E2074/local-datasets/customer/ads-monetisation/metadata


<div class="alert alert-block alert-warning">  
    
## Datasets & Functions
</div>

In [6]:
start = datetime.strptime(start_date, '%Y%m%d')
end = datetime.strptime(end_date, '%Y%m%d')

# Generate the list of dates
date_range = [(start + timedelta(days=i)).strftime('%Y%m%d') for i in range((end - start).days + 1)]
print(date_range)

['20250101', '20250102', '20250103', '20250104', '20250105', '20250106', '20250107', '20250108', '20250109', '20250110', '20250111', '20250112', '20250113', '20250114', '20250115', '20250116', '20250117', '20250118', '20250119', '20250120', '20250121']


In [7]:
def get_customer_metadata(date_range, city):

    all_df = []

    title_pattern = 'title[=:]"?([^",}]*)'
    body_pattern = 'body[=:]"?([^",}]*)'
    advertiserName_pattern = 'advertiserName[=:]"?([^",}]*)'
    
    for date in date_range:
    
        customer_metadata = f"""
        
            WITH metadata as (
            SELECT
                userid,
                title,
                body,
                advertiserName
            FROM 
                (
                SELECT
                    yyyymmdd,
                    userid,
                    -- aduuid,
                    length(metadata1) metabase_length,
                    
                    CASE 
                    WHEN os = 'android' THEN regexp_extract(metadata1, '{title_pattern}' , 1) 
                    WHEN os = 'iOS' THEN json_extract_scalar(metadata1, '$.title') 
                    END AS title,
                    
                    CASE 
                    WHEN os = 'android' THEN regexp_extract(metadata1, '{body_pattern}', 1)
                    WHEN os = 'iOS' THEN json_extract_scalar(metadata1, '$.body')
                    END AS body,
                    
                    CASE 
                    WHEN os = 'android' THEN regexp_extract(metadata1, '{advertiserName_pattern}', 1)
                    WHEN os = 'iOS' THEN json_extract_scalar(metadata1, '$.advertiserName')
                    END AS advertiserName
                    
                    
                FROM 
                    canonical.iceberg_log_telemetry_ads_impressions_immutable
                WHERE 
                    yyyymmdd = '{date}'
                    AND city = '{city}'
                    AND responseType = 'GAMBanner'
                    AND os = 'android'
                    AND eventName = 'Ad_Response'
                    AND length(metadata1) > 5
                )
            WHERE
                (title IS NOT NULL OR body IS NOT NULL OR advertiserName IS NOT NULL)
            )
            
            SELECT
                *
            FROM 
                metadata
            
        """
        
        df = pd.read_sql(customer_metadata, connection)
        all_df.append(df)
        df.to_parquet(local_dataset + '/data_dump/response_metadata_{}.parquet'.format(date), index=False)
    
    # Concatenate all DataFrames
    final_df = pd.concat(all_df, ignore_index=True)

    return final_df


df_data = get_customer_metadata(date_range, city)

In [8]:
def get_customer_base(start_date, end_date, city):
    
    customer_metadata = f"""
    
        with customers as (

            select
                distinct 
                customer_id,
                case 
                when customer_obj_gender = '0' then 'Male' 
                when customer_obj_gender = '1' then 'Female' 
                else 'Other/Unknow' end gender
            from 
                orders.order_logs_fact
            where 
                yyyymmdd >= '{start_date}'
                and yyyymmdd <= '{end_date}'
                and city_name = 'Bangalore'
            ),
            
            segment as (
        
            select
                customer_id,
                taxi_ltr_segment,
                taxi_retention_segments,
                taxi_income_segment,
                ps_tag_link,
                ps_tag_auto,
                taxi_regularity_segment
            from 
                datasets.iallocator_customer_segments
            where 
                run_date = '2025-01-21'
            ),
            
            
            app_cat as (
            
            select 
                *
            from 
                reports.sql_ingestion_customer_appography_category_view
            where 
                yyyymmdd = '20250112'
            )
            
            
            select 
                customers.*,
                taxi_ltr_segment,
                taxi_retention_segments,
                taxi_regularity_segment,
                taxi_income_segment,
                ps_tag_link,
                ps_tag_auto,
                ai,
                camerafilter,
                courier,
                dating,
                deliveryfood,
                deliverygrocery,
                devotional,
                driverapp,
                ecommerce,
                educational,
                entertainment,
                fantasysports,
                financeinvestment,
                financenews,
                financetransactions,
                gaming,
                government,
                health,
                homesecurity,
                insurance,
                jobsearch,
                news,
                office,
                ott,
                social,
                streamingmusic,
                telecom,
                tools,
                travelbookings,
                travelmetro,
                travelridehailing,
                vehicle,
                wearable
            from 
                customers
            join 
                app_cat
                on customers.customer_id = app_cat.userid
            left join 
                segment
                on customers.customer_id = segment.customer_id
        
    """
        
    df = pd.read_sql(customer_metadata, connection)
    df.to_parquet(local_dataset + '/gross_customers.parquet', index=False)
    
    return df


df_customer_base = get_customer_base(start_date, end_date, city)

NameError: name 'final_df' is not defined

In [8]:
def get_customer_metadata():
    
    # Use glob to find all Parquet files in the directory
    parquet_files = glob.glob(os.path.join(local_dataset + '/data_dump/', "*.parquet"))
    df = pd.concat([pd.read_parquet(file) for file in parquet_files], ignore_index=True)
    
    return df

df_data = get_customer_metadata()

In [9]:
# Generate a unique hash identifier for each combination
df_data["hash_id"] = df_data.apply(lambda row: hashlib.md5(f"{row['title']}_{row['body']}_{row['advertiserName']}".encode()).hexdigest(), axis=1)
df_data["metadata"] = df_data.apply(lambda row: f"advertiserName: {row['advertiserName']}, title: {row['title']}, body: {row['body']}", axis=1)

In [10]:
df_data.head(1)

,userid,title,body,advertiserName,hash_id,metadata
0,6357d1c63c9d5440e8d60f6c,Get offer of Rs.16000 on V STROM Bike,Limited Stock,Suzuki,e90df072d8ceba79c8b1bc0641015e87,"advertiserName: Suzuki, title: Get offer of Rs..."


In [11]:
df_metadata = df_data[["hash_id", "advertiserName", "title", "body", "metadata"]].drop_duplicates()

print(df_data.shape)
print(df_metadata.shape)
df_metadata.head(1)

(728646, 6)
(4440, 5)


,hash_id,advertiserName,title,body,metadata
0,e90df072d8ceba79c8b1bc0641015e87,Suzuki,Get offer of Rs.16000 on V STROM Bike,Limited Stock,"advertiserName: Suzuki, title: Get offer of Rs..."


<div class="alert alert-block alert-success">  
    
## Analysis
</div>

In [12]:
# Load the pre-trained model and tokenizer
classifier = pipeline("zero-shot-classification", model="valhalla/distilbart-mnli-12-1")

# Define categories
categories = [
    'Finance','Technology','Entertainment','Automotive','Healthcare','Retail/E-commerce','Travel & Tourism',
    'Food & Beverage', 'Real Estate','Education','Beauty & Personal Care','Consumer Goods','Telecommunications',
    'Sports & Recreation', 'Social Network & Media','Fashion & Apparel','Home & Garden','Legal Services',
    'Logistics & Delivery Services', 'Gaming & Esports','Media & Publishing','Business Services','Energy & Utilities',
    'Agriculture','Luxury Goods & Services', 'Events & Conferences','Parenting & Childcare','Pets & Animals',
    'DIY & Crafts','Spirituality & Religion', 'Automobile', 'Banking', 'Investments', 'Trading', 'Unknown'
]

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


In [13]:
df_metadata.head(1)

,hash_id,advertiserName,title,body,metadata
0,e90df072d8ceba79c8b1bc0641015e87,Suzuki,Get offer of Rs.16000 on V STROM Bike,Limited Stock,"advertiserName: Suzuki, title: Get offer of Rs..."


In [ ]:
start_time = time.time()

# Function to classify text and add category columns
def classify_text(input_data, classifier, categories):
    if isinstance(input_data, str):  # Handle single-column input
        text = input_data or ''
        return classifier(text, candidate_labels=categories)['labels'][0] if text.strip() else 'Unknown'
    else:  # Handle row input
        categories_dict = {}
        for field in ['advertiserName', 'title', 'body']:
            text = input_data[field] or ''
            categories_dict[f'category_{field}'] = (
                classifier(text, candidate_labels=categories)['labels'][0]
                if text.strip() else 'Unknown'
            )
        return pd.Series(categories_dict)

def get_classification_tag(df):
    # Fill missing values in the necessary columns
    df[['advertiserName', 'title', 'body']] = df[['advertiserName', 'title', 'body']].fillna('')

    # Initialize category columns with default values
    df['category_advertiserName'] = df['advertiserName'].apply(lambda x: classify_text(x, classifier, categories) if x else 'Unknown')
    df['category_title'] = 'Unknown'  # Set default values
    df['category_body'] = 'Unknown'

    # Apply classification for title and body only if advertiserName category is 'Unknown'
    mask_title = df['category_advertiserName'] == 'Unknown'
    df.loc[mask_title, 'category_title'] = df.loc[mask_title, 'title'].apply(
        lambda x: classify_text(x, classifier, categories) if x else 'Unknown'
    )

    mask_body = (df['category_advertiserName'] == 'Unknown') & (df['category_title'] == 'Unknown')
    df.loc[mask_body, 'category_body'] = df.loc[mask_body, 'body'].apply(
        lambda x: classify_text(x, classifier, categories) if x else 'Unknown'
    )

    # Combine categories based on conditions
    df['category'] = df.apply(
        lambda row: row['category_advertiserName'] if row['category_advertiserName'] != 'Unknown'
        else (row['category_title'] if row['category_title'] != 'Unknown' else row['category_body']),
        axis=1
    )

    return df[['hash_id', 'advertiserName', 'title', 'body', 'category']]



df_category = get_classification_tag(df_metadata)

df_category.to_parquet(local_dataset + '/hash_id_category.parquet', index=False)

end_time = time.time()
execution_time = end_time - start_time
print(f"Execution time: {execution_time} seconds")

In [ ]:
df_category.head(1)

In [ ]:
df_merge = pd.merge(df_data, df_category[['hash_id', 'inner']], how='left', on ='hash_id')

In [ ]:
df_merge.head(1)